# Temporal Patient Journey — Tracking Decisions Over Time

**Follow ONE patient across 3 clinical timepoints as the CAP CDSS agent adapts its recommendations.**

> This notebook is generated by `_generate_temporal_journey_notebook.py`. Do not edit manually.

### Clinical Trajectory

| Timepoint | Phase | Key Decision |
|-----------|-------|-------------|
| **T=0** | Admission | Initial severity assessment, antibiotic selection, admission decision |
| **T=48h** | Review | CRP trend evaluation, IV-to-oral switch eligibility, treatment response |
| **Day 3-4** | Discharge | 7 discharge criteria gates, place-of-care re-evaluation, treatment duration |

This notebook demonstrates **temporal reasoning** — the hallmark of an agentic system.
Rather than analysing isolated snapshots, the pipeline tracks clinical trajectories and
adapts its recommendations as the patient's condition evolves.

Each timepoint runs the full 8-node LangGraph pipeline with updated clinical data,
and the agent's severity assessment, treatment pathway, and discharge recommendation
evolve accordingly.

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **GITHUB_TOKEN** for private repo install
4. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/CAP-CDSS-MedGemma.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/CAP-CDSS-MedGemma.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 pandas>=2.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."
print("Authentication complete")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph
from cap_agent.agent.state import build_initial_state

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## Pipeline Overview: 3 Timepoints

This notebook runs the **same patient** through 3 distinct clinical scenarios, each
representing a different point in their hospital admission:

### T=0: Admission
- **Patient:** 72-year-old male, COPD, Type 2 Diabetes
- **Presentation:** Productive cough, fever 38.5C, dyspnoea
- **Data:** Full FHIR bundle (18 observations) + clerking note + lab report
- **GPU calls:** Up to 9 (3 EHR + 2 Lab + 3 CXR + 1 contradiction/summary)

### T=48h: Review
- **Update:** CRP trending down (186 -> 95 mg/L), vitals improving
- **Key question:** Is the treatment working? IV-to-oral switch?
- **Stewardship rules:** CR-7 through CR-11 evaluated with temporal context

### Day 3-4: Discharge Assessment
- **Update:** CRP continuing to improve, stability markers assessed
- **Key question:** Does the patient meet all 7 discharge criteria?
- **Temporal reasoning:** Treatment response drives discharge decision


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


## Timepoint 1: Admission (T=0)

**Patient:** 72-year-old male presenting with productive cough, fever (38.5C), and dyspnoea.

**Past medical history:** COPD, Type 2 Diabetes

This is the baseline assessment — the agent receives the full clinical picture:
- **FHIR bundle** with 18 observations (vitals, labs, functional status)
- **Clerking note** with history, examination, and impression
- **Lab report** with full blood panel

The pipeline uses **real EHR QA extraction** (3 GPU calls) and **real Lab extraction** (2 GPU calls)
to parse these structured and unstructured data sources.


In [ ]:
from cap_agent.data.synthetic import get_synthetic_case
from cap_agent.agent.state import build_initial_state

t0_case = get_synthetic_case()
initial_state = build_initial_state(t0_case)
print(f"Case: {t0_case['case_id']}")
print(f"Patient: {t0_case['demographics']['age']}yo {t0_case['demographics']['sex']}")
print(f"FHIR bundle: {'Yes' if t0_case.get('fhir_bundle') else 'No'}")
print(f"Lab report: {'Yes' if t0_case.get('lab_report') else 'No'}")
print(f"Clerking note: {'Yes' if t0_case.get('clerking_note') else 'No'}")
print()

t0_result = None
start = time.time()
for chunk in graph.stream(initial_state, stream_mode="values"):
    t0_result = chunk
t0_elapsed = time.time() - start
print(f"Pipeline complete in {t0_elapsed:.1f}s")


In [ ]:
# Extract key metrics
t0_curb = t0_result.get("curb65_score", {})
t0_abx = t0_result.get("antibiotic_recommendation", {})
t0_labs = t0_result.get("lab_values", {})
t0_monitor = t0_result.get("monitoring_plan", {})

print("=== ADMISSION SNAPSHOT ===")
print(f"CURB65: {t0_curb.get('curb65')} — Severity: {t0_curb.get('severity_tier', '?').upper()}")
print(f"  C={t0_curb.get('c',0)} U={t0_curb.get('u',0)} R={t0_curb.get('r',0)} B={t0_curb.get('b',0)} 65={t0_curb.get('age_65',0)}")
print(f"\nCRP: {t0_labs.get('crp', '?')} mg/L")
print(f"Antibiotic: {t0_abx.get('first_line', '?')}")
print(f"Route: {t0_abx.get('route', '?')}")
print(f"\nContradictions detected: {len(t0_result.get('contradictions_detected', []))}")
for c in t0_result.get("contradictions_detected", []):
    print(f"  {c.get('rule_id', '?')}: {c.get('description', '?')}")

# Discharge criteria at admission
so = t0_result.get("structured_output", {})
sections = so.get("sections", {})
s7 = sections.get("7_monitoring", {})
discharge = s7.get("discharge_criteria", {})
print(f"\nDischarge criteria met: {discharge.get('all_met', 'N/A')}")


In [ ]:
render_clinical_output(t0_result, "Admission (T=0)")


## Timepoint 2: 48-Hour Review

**48 hours into treatment** — clinical reassessment with updated data.

Key changes from admission:
- **CRP trending:** 186 -> 95 mg/L (approximately 49% reduction — "slow response")
- **Vitals improving** but still requiring monitoring
- **Treatment reassessment:** Is IV-to-oral switch appropriate?

The pipeline evaluates:
- CRP trend classification (improving / slow / static / worsening)
- Treatment response assessment
- Stewardship rules (CR-7 through CR-11) with temporal context
- IV-to-oral switch eligibility (CR-9)


In [ ]:
from cap_agent.data.synthetic import get_synthetic_48h_case

t48_case = get_synthetic_48h_case()
initial_state = build_initial_state(t48_case)
print(f"Case: {t48_case['case_id']}")
print(f"Treatment status: {t48_case.get('treatment_status', {}).get('current_antibiotic', '?')}")
print(f"Admission CRP: {t48_case.get('admission_labs', {}).get('crp', '?')} mg/L")
print()

t48_result = None
start = time.time()
for chunk in graph.stream(initial_state, stream_mode="values"):
    t48_result = chunk
t48_elapsed = time.time() - start
print(f"Pipeline complete in {t48_elapsed:.1f}s")


In [ ]:
t48_curb = t48_result.get("curb65_score", {})
t48_abx = t48_result.get("antibiotic_recommendation", {})
t48_labs = t48_result.get("lab_values", {})

print("=== 48-HOUR REVIEW ===")
print(f"CURB65: {t48_curb.get('curb65')} — Severity: {t48_curb.get('severity_tier', '?').upper()}")
print(f"\nCRP: {t48_labs.get('crp', '?')} mg/L (admission: {t48_case.get('admission_labs', {}).get('crp', '?')} mg/L)")
print(f"Antibiotic: {t48_abx.get('first_line', '?')}")
print(f"\nContradictions detected: {len(t48_result.get('contradictions_detected', []))}")
for c in t48_result.get("contradictions_detected", []):
    print(f"  {c.get('rule_id', '?')}: {c.get('description', '?')}")

# CRP trend
so = t48_result.get("structured_output", {})
sections = so.get("sections", {})
s4 = sections.get("4_labs", {})
crp_trend = s4.get("crp_trend", {})
print(f"\nCRP trend: {crp_trend.get('trend_label', '?')} ({crp_trend.get('percentage_change', '?')}% change)")

# Discharge
s7 = sections.get("7_monitoring", {})
discharge = s7.get("discharge_criteria", {})
print(f"Discharge criteria met: {discharge.get('all_met', 'N/A')}")


In [ ]:
render_clinical_output(t48_result, "48-Hour Review")


## Timepoint 3: Day 3-4 Discharge Assessment

**Day 3-4 of treatment** — formal discharge assessment.

The pipeline evaluates:
- **CRP continuing to improve** from the 48h value
- **7 discharge criteria gates** — all must pass for discharge recommendation
- **Treatment response** — determines whether to continue, step down, or discharge
- **Place-of-care re-evaluation** — can the patient be safely managed at home?

The 7 discharge gates:
1. Temperature < 37.5C for 24h
2. Heart rate < 100 bpm
3. Respiratory rate < 24/min
4. SBP >= 90 mmHg
5. SpO2 >= 90% on room air
6. Tolerating oral medication and intake
7. Mental status returned to baseline


In [ ]:
from cap_agent.data.synthetic import get_synthetic_day34_case

d34_case = get_synthetic_day34_case()
initial_state = build_initial_state(d34_case)
print(f"Case: {d34_case['case_id']}")
print(f"Treatment status: {d34_case.get('treatment_status', {}).get('current_antibiotic', '?')}")
print(f"Admission CRP: {d34_case.get('admission_labs', {}).get('crp', '?')} mg/L")
print()

d34_result = None
start = time.time()
for chunk in graph.stream(initial_state, stream_mode="values"):
    d34_result = chunk
d34_elapsed = time.time() - start
print(f"Pipeline complete in {d34_elapsed:.1f}s")


In [ ]:
d34_curb = d34_result.get("curb65_score", {})
d34_abx = d34_result.get("antibiotic_recommendation", {})
d34_labs = d34_result.get("lab_values", {})

print("=== DAY 3-4 DISCHARGE ASSESSMENT ===")
print(f"CURB65: {d34_curb.get('curb65')} — Severity: {d34_curb.get('severity_tier', '?').upper()}")
print(f"\nCRP: {d34_labs.get('crp', '?')} mg/L")
print(f"Antibiotic: {d34_abx.get('first_line', '?')}")
print(f"\nContradictions detected: {len(d34_result.get('contradictions_detected', []))}")
for c in d34_result.get("contradictions_detected", []):
    print(f"  {c.get('rule_id', '?')}: {c.get('description', '?')}")

# Treatment response
so = d34_result.get("structured_output", {})
sections = so.get("sections", {})
s7 = sections.get("7_monitoring", {})
treatment = s7.get("treatment_response", {})
print(f"\nTreatment response: {treatment.get('assessment', '?')}")
print(f"Reassessment needed: {treatment.get('reassess_needed', '?')}")

# Discharge criteria
discharge = s7.get("discharge_criteria", {})
print(f"\nDischarge criteria met: {discharge.get('all_met', 'N/A')}")
criteria = discharge.get("criteria", {})
for k, v in criteria.items():
    status = "PASS" if v else "FAIL"
    color = "" if v else "  <- BLOCKING"
    print(f"  {k}: {status}{color}")


In [ ]:
render_clinical_output(d34_result, "Day 3-4 Discharge Assessment")


## Clinical Evolution: Delta Visualizations

Compare key metrics across all 3 timepoints:
- **CRP trend** with clinical response thresholds
- **CURB65 component heatmap** showing which parameters change over time
- **Treatment pathway evolution** from admission through discharge


In [ ]:
import plotly.graph_objects as go

# Collect CRP values across timepoints
timepoints = ["T=0 (Admission)", "T=48h (Review)", "Day 3-4 (Discharge)"]
crp_values = [
    t0_labs.get("crp", 0),
    t48_labs.get("crp", 0),
    d34_labs.get("crp", 0),
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=timepoints, y=crp_values,
    mode="lines+markers+text",
    text=[f"{v} mg/L" for v in crp_values],
    textposition="top center",
    line=dict(color="#2563eb", width=3),
    marker=dict(size=12),
    name="CRP"
))

# Add threshold annotations
if crp_values[0] > 0:
    # 50% reduction line (improving)
    fifty_pct = crp_values[0] * 0.5
    fig.add_hline(y=fifty_pct, line_dash="dash", line_color="green",
                  annotation_text=f">=50% reduction = Improving ({fifty_pct:.0f})")
    # 75% of original (slow response threshold)
    seventy_five = crp_values[0] * 0.75
    fig.add_hline(y=seventy_five, line_dash="dash", line_color="orange",
                  annotation_text=f"25-49% reduction = Slow ({seventy_five:.0f})")

fig.update_layout(
    title="CRP Trend Across Timepoints",
    yaxis_title="CRP (mg/L)",
    template="plotly_white",
    height=400,
)
fig.show()


In [ ]:
import plotly.graph_objects as go

components = ["C (Confusion)", "U (Urea)", "R (Resp Rate)", "B (Blood Pressure)", "65 (Age)"]
t0_vals = [t0_curb.get("c", 0), t0_curb.get("u", 0), t0_curb.get("r", 0), t0_curb.get("b", 0), t0_curb.get("age_65", 0)]
t48_vals = [t48_curb.get("c", 0), t48_curb.get("u", 0), t48_curb.get("r", 0), t48_curb.get("b", 0), t48_curb.get("age_65", 0)]
d34_vals = [d34_curb.get("c", 0), d34_curb.get("u", 0), d34_curb.get("r", 0), d34_curb.get("b", 0), d34_curb.get("age_65", 0)]

fig = go.Figure(data=go.Heatmap(
    z=[t0_vals, t48_vals, d34_vals],
    x=components,
    y=["T=0", "T=48h", "Day 3-4"],
    colorscale=[[0, "#dcfce7"], [1, "#dc2626"]],
    text=[[str(v) for v in row] for row in [t0_vals, t48_vals, d34_vals]],
    texttemplate="%{text}",
    showscale=False,
))

fig.update_layout(
    title="CURB65 Components Across Timepoints",
    template="plotly_white",
    height=300,
)
fig.show()


In [ ]:
import plotly.graph_objects as go

treatments = [
    t0_abx.get("first_line", "?"),
    t48_abx.get("first_line", "?"),
    d34_abx.get("first_line", "?"),
]
routes = [
    t0_abx.get("route", "?"),
    t48_abx.get("route", "?"),
    d34_abx.get("route", "?"),
]
severities = [
    t0_curb.get("severity_tier", "?"),
    t48_curb.get("severity_tier", "?"),
    d34_curb.get("severity_tier", "?"),
]

print("=== TREATMENT PATHWAY EVOLUTION ===")
print(f"{'Timepoint':<25} {'Severity':<12} {'Route':<8} {'Antibiotic'}")
print("-" * 80)
for tp, sev, route, abx in zip(timepoints, severities, routes, treatments):
    print(f"{tp:<25} {sev:<12} {route:<8} {abx}")


## Summary: What the Agent Learned Over Time

### Temporal Reasoning in Practice

- **At T=0 (Admission):** The agent identified a high-severity CAP case requiring IV antibiotics
  and hospital admission. It parsed the full FHIR bundle, clerking note, and lab report to build
  a comprehensive clinical picture including CURB65 scoring and treatment selection.

- **At T=48h (Review):** The agent reassessed CRP trend using clinical thresholds, evaluated
  treatment response, and checked IV-to-oral switch eligibility via CR-9. It compared current
  labs against admission values to classify the CRP trajectory.

- **At Day 3-4 (Discharge):** The agent ran the full discharge criteria checklist (7 binary gates),
  assessed treatment response, and made a place-of-care recommendation. Treatment reassessment
  can override discharge — if the patient is not responding, discharge is blocked regardless of
  vital sign stability.

### Why This Matters

This temporal reasoning is a hallmark of **agentic systems**. The pipeline does not just analyse
isolated clinical snapshots — it tracks clinical trajectories over time and adapts its
recommendations accordingly. Each timepoint uses the same 8-node pipeline, but the clinical
data evolves, and the agent's decisions evolve with it.

This mirrors real clinical workflow: admission assessment, treatment review, and discharge
planning are distinct decision points that build on each other.


In [ ]:
print("=== VERIFICATION CHECKLIST ===\n")

checks = [
    ("T=0 CURB65 = 3 (high severity)", t0_curb.get("curb65") == 3),
    ("T=0 severity = high", t0_curb.get("severity_tier") == "high"),
    ("T=0 CRP > 100", (t0_labs.get("crp") or 0) > 100),
    ("T=0 discharge NOT met", not t0_result.get("structured_output", {}).get("sections", {}).get("7_monitoring", {}).get("discharge_criteria", {}).get("all_met", True)),
    ("T=48h pipeline ran successfully", t48_result is not None),
    ("T=48h has CRP trend data", bool(t48_result.get("structured_output", {}).get("sections", {}).get("4_labs", {}).get("crp_trend"))),
    ("Day 3-4 pipeline ran successfully", d34_result is not None),
    ("Day 3-4 has treatment response", bool(d34_result.get("structured_output", {}).get("sections", {}).get("7_monitoring", {}).get("treatment_response"))),
    ("Day 3-4 has discharge criteria", bool(d34_result.get("structured_output", {}).get("sections", {}).get("7_monitoring", {}).get("discharge_criteria"))),
]

passed = 0
for desc, ok in checks:
    status = "PASS" if ok else "FAIL"
    icon = "+" if ok else "x"
    print(f"  {icon} {status}: {desc}")
    if ok:
        passed += 1

print(f"\n{passed}/{len(checks)} checks passed")
